# RegionAnalyzer — synthetic 3D segmentation

Build a small labeled volume (three non-overlapping objects) and extract region properties as a pandas DataFrame.

In [32]:
import numpy as np
import pandas as pd
import stackview

from vistiq.utils import ArrayIteratorConfig
from vistiq.constant.matrix import FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, MinFilterConfig, RangeFilterConfig
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig, MatrixAggregator, MatrixAggregatorConfig
from vistiq.analysis import DistanceCalculator, DistanceCalculatorConfig

## Synthetic label volume

Shape `(100, 100, 10)` with labels `1`, `2`, and `3` in separate spatial regions (no overlap), mimicking a 3D segmentation mask.

In [33]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 42:58, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:58] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [34]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6


# Area2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [35]:
stackview.slice(np.concatenate([labels, areas], axis=-1))

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [36]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=metadata)

2026-06-10 08:04:18,181 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:18,250 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:18,268 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'stack_id', 'slice_id', 'object_id', 'centroid', 'bbox', 'aspect_ratio', 'cross_sectional_area', 'area'] map_axes=True expand_co

In [37]:
l_regions

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,f74970bae3e740aa90f278816b7a166e,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,c69c9280b5874d2c88fcaeda72b904bc,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,8bea06e071fa402c88c12167f9b14f12,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,899abf27161e4d6aa285674a4f5f5f76,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,8fa829f3ebb6486299eb5d85d7405e45,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b


In [38]:
a_regions

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,f44aed4f9a714654bf4cb964c129a6aa,91fe6b573c9b491d8507b71ff659e31d,82281ff6b11c40fe88ad4408e74d9350
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,2c1552d8d0934567bad49483e299bd82,91fe6b573c9b491d8507b71ff659e31d,82281ff6b11c40fe88ad4408e74d9350


# Filter Regions

In [39]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(100.0,np.inf)
        ),
        MinFilterConfig(
            attribute="aspect_ratio",
            minimum=0.015,
        ),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)

2026-06-10 08:04:18,975 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:19,040 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:19,048 - INFO - Running RegionFilter with config: classname='Configurable' package='vistiq.core' version=None command_group=None filters=[RangeFilterConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, attribute='volume', axis=None, strict=True, preferred_input_type='numpy', range=(100.0, inf)), MinFilterConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, attribute='aspect_ratio', axis=None, strict=True, preferred_input_type='numpy', minimum=0.015, operator='gte')]
2026-06-10

In [40]:
l_accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,f74970bae3e740aa90f278816b7a166e,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,c69c9280b5874d2c88fcaeda72b904bc,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,8bea06e071fa402c88c12167f9b14f12,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,899abf27161e4d6aa285674a4f5f5f76,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,8fa829f3ebb6486299eb5d85d7405e45,2a9dde4445694f3499b511d610cf32cb,8897754dc8d544a9b93b8aa1925fa50b


In [41]:
a_accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,f44aed4f9a714654bf4cb964c129a6aa,91fe6b573c9b491d8507b71ff659e31d,82281ff6b11c40fe88ad4408e74d9350
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,2c1552d8d0934567bad49483e299bd82,91fe6b573c9b491d8507b71ff659e31d,82281ff6b11c40fe88ad4408e74d9350


# Calculate inter-object distances

Uses PyTorch tensors. The config allows setting a `preferred_device` ("cuda", "mps", "cpu", None). This is not a guarantee. The actual device can be assigned at runtime with `device`. If the device is None, it will be auto-discovered considering the config's `preferred_device`.

In [42]:
dccfg = DistanceCalculatorConfig(
    annotate=True, 
    output_type="torch.Tensor",
    preferred_device="cuda",
)

centroids = dataframe_to_numpy(l_accepted, attributes=["centroid"], strict=False)
object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(
    centroids, 
    centroids, 
    spacing=metadata.get("scale", None), 
    point_annotations=(object_ids, object_ids),
    device=None
)

2026-06-10 08:04:20,071 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:20,121 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 08:04:20,127 - WARNING - Preferred device 'cuda' unavailable; falling back via check_device()
2026-06-10 08:04:20,128 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 08:04:20,133 - INFO - MatrixCalculator.run: device=mps, points1.shape=torch.Size([5, 3]), points1.dtype=torch.float32, points2.shape=torch.Size([5, 3]), points2.dtype=torch.float32, spacing=(2.0, 1.0, 1.0)
2026-06-10 08:04:20,134 - INFO - DistanceCalculator._calculate: distances.shape=torch.Size([5, 5])
2026-06-10 08:04:20,136 - INFO - Finished in state Completed()


In [43]:
type(dist), dist

(torch.Tensor,
 tensor([[  0.0000,  36.3043,  77.9359, 154.2141, 134.9602],
         [ 36.3043,   0.0000,  43.6807, 118.4736, 113.7772],
         [ 77.9359,  43.6807,   0.0000,  77.4080,  93.0229],
         [154.2141, 118.4736,  77.4080,   0.0000, 116.8985],
         [134.9602, 113.7772,  93.0229, 116.8985,   0.0000]], device='mps:0'))

# Apply a Rank Filter

`axis=0`: column-wise
`axis=1`: row-wise
`axis=None`: global

`output` options: ["masked_values", "indices", "mask", "values"]

In [44]:
tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist)
tk

2026-06-10 08:04:21,058 - INFO - Found mps device: Apple Metal (MPS)


tensor([[    nan, 36.3043,     nan,     nan,     nan],
        [36.3043,     nan,     nan,     nan,     nan],
        [    nan, 43.6807,     nan,     nan,     nan],
        [    nan,     nan, 77.4080,     nan,     nan],
        [    nan,     nan, 93.0229,     nan,     nan]], device='mps:0')

# Apply a Threshold based Filter

In [45]:
mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)
maxcfg = ValueFilterConfig(
    ref_value=120.0,
    axis=0,
    operator="<",
    triangle=LOWER_ND,
    output="masked_values",
)
mint = ValueFilter(mincfg).run(dist)
mint

2026-06-10 08:04:21,610 - INFO - Found mps device: Apple Metal (MPS)


tensor([[     nan,      nan,      nan,      nan,      nan],
        [     nan,      nan,      nan,      nan,      nan],
        [     nan,      nan,      nan,      nan,      nan],
        [154.2141, 118.4736,      nan,      nan,      nan],
        [134.9602, 113.7772,  93.0229, 116.8985,      nan]], device='mps:0')

# Aggregate

In [48]:
macfg = MatrixAggregatorConfig(
    operation="count",
    axis=1,
)

counts = MatrixAggregator(macfg).run(mint)
counts

2026-06-10 08:05:09,980 - INFO - Found mps device: Apple Metal (MPS)


tensor([0, 0, 0, 2, 4], device='mps:0')